In [7]:
import numpy as np
import pandas as pd

# 1. Load the *raw combined* dataset (after fixing label mapping in load_promise_nasa.py)
combined_path = "../../data/processed/promise_nasa_combined.csv"
df = pd.read_csv(combined_path)

print("Original combined shape:", df.shape)

# Quick sanity check of label balance BEFORE cleaning
print("\nLabel distribution before cleaning:")
print(df["label"].value_counts(normalize=True).rename("fraction"))
print("\nModules per dataset:")
print(df["dataset"].value_counts())

Original combined shape: (13492, 23)

Label distribution before cleaning:
label
0    0.816113
1    0.183887
Name: fraction, dtype: float64

Modules per dataset:
dataset
JM1    10885
KC1     2109
CM1      498
Name: count, dtype: int64


In [8]:
clean = df.copy()

# 2. Drop exact duplicate rows only (no other row-wise drops)
before = clean.shape[0]
clean = clean.drop_duplicates()
after = clean.shape[0]
print(f"Dropped {before - after} duplicate rows")

# 3. Replace inf / -inf with NaN in numeric columns
num_cols = clean.select_dtypes(include=["number"]).columns
clean[num_cols] = clean[num_cols].replace([np.inf, -np.inf], np.nan)

# 4. Drop columns that are entirely missing
cols_before = set(clean.columns)
clean = clean.dropna(axis=1, how="all")
dropped_all_na = cols_before - set(clean.columns)
print("Dropped all-NaN columns:", list(dropped_all_na))

# 5. Drop only *extremely* sparse numeric columns (e.g. > 80% missing)
num_cols = clean.select_dtypes(include=["number"]).columns
missing_frac = clean[num_cols].isna().mean()
high_missing = missing_frac[missing_frac > 0.8].index.tolist()
print("Numeric columns with >80% missing (to drop):", high_missing)

clean = clean.drop(columns=high_missing)
num_cols = clean.select_dtypes(include=["number"]).columns

Dropped 2926 duplicate rows
Dropped all-NaN columns: []
Numeric columns with >80% missing (to drop): []


In [9]:
# 6. Median-impute remaining numeric NaNs (do NOT drop rows)
median_values = clean[num_cols].median()
clean[num_cols] = clean[num_cols].fillna(median_values)

# 7. Remove constant (single-value) numeric columns
nunique = clean[num_cols].nunique()
constant_cols = nunique[nunique <= 1].index.tolist()
print("Constant numeric columns dropped:", constant_cols)

clean = clean.drop(columns=constant_cols)

print("Cleaned shape:", clean.shape)

Constant numeric columns dropped: []
Cleaned shape: (10566, 23)


In [10]:
print("\nLabel distribution AFTER cleaning:")
print(clean["label"].value_counts(normalize=True).rename("fraction"))

print("\nModules per dataset AFTER cleaning:")
print(clean["dataset"].value_counts())


Label distribution AFTER cleaning:
label
0    0.775696
1    0.224304
Name: fraction, dtype: float64

Modules per dataset AFTER cleaning:
dataset
JM1    8912
KC1    1212
CM1     442
Name: count, dtype: int64


In [11]:
output_path = "../../data/processed/promise_nasa_combined_clean.csv"
clean.to_csv(output_path, index=False)

print(f"\nSaved cleaned dataset to: {output_path}")
print("Final columns:", clean.columns.tolist())


Saved cleaned dataset to: ../../data/processed/promise_nasa_combined_clean.csv
Final columns: ['loc', 'v(g)', 'ev(g)', 'iv(g)', 'n', 'v', 'l', 'd', 'i', 'e', 'b', 't', 'lOCode', 'lOComment', 'lOBlank', 'locCodeAndComment', 'uniq_Op', 'uniq_Opnd', 'total_Op', 'total_Opnd', 'branchCount', 'label', 'dataset']


In [12]:
# 5. Impute remaining NaNs in numeric columns with median
median_values = clean[num_cols].median()
clean[num_cols] = clean[num_cols].fillna(median_values)

# 6. Remove constant (and near-constant) numeric columns
nunique = clean[num_cols].nunique()
constant_cols = nunique[nunique <= 1].index.tolist()
print("Constant / single-value numeric columns dropped:", constant_cols)

clean = clean.drop(columns=constant_cols)

print("Cleaned shape:", clean.shape)

Constant / single-value numeric columns dropped: []
Cleaned shape: (10566, 23)
